# Distillation: ConvNeXt-Tiny taught by the CNN + RAD-DINO average (NIH official split, seed 42)

**Question.** Can a ConvNeXt-Tiny trained on the soft predictions of the CNN + RAD-DINO average reach RAD-DINO-level accuracy at the CNN's inference cost (0.18x RAD-DINO at batch 32)? RAD-DINO is used at training time only.

**Rules:** `evaluation_protocol.yaml` v2.2.0, section `distillation`, written before any distillation result. The cascade notebook (`2-staged_official_split.ipynb`) is not changed.

| Session | Flag | Hardware | Time (T4) | Output |
|---|---|---|---|---|
| D1 | `RUN_TEACHER` | GPU | ~45 min | teacher QC (D1-D3), `soft_labels_train.npz`, `distill_go_no_go.json` |
| D2 | `RUN_STUDENT` | GPU | ~3 h | run folder `..._M10_CONVNEXT_TINY_KD_384__s42` (checkpoint, val/test logits), QC-D4 |
| D3 | - | CPU | minutes | evaluation (to be added after D2) |

D1 and D2 can run in one GPU session: D2 starts only if the go/no-go rule says **go**.

**Inputs to attach on Kaggle**
1. `nih-chest-xrays/data` (images, `Data_Entry_2017.csv`, official list files).
2. A dataset with `results_s42/` (at least `calibration.json` and the two `runs/` folders with their logits), and each teacher's `checkpoint_best.pt` placed inside its run folder:
   - `NIH_CXR14__NIH_OFFICIAL_PV__M08_CONVNEXT_TINY_ASL_384_CR__be576650cff5__s42__r0/checkpoint_best.pt` (sha256 `eebd5f66...`)
   - `NIH_CXR14__NIH_OFFICIAL_PV__M09_RAD_DINO_518_CR__cb7d09842184__s42__r0/checkpoint_best.pt` (sha256 `070f90c7...`, **not** `checkpoint_last.pt`)
3. Internet on (ImageNet weights for the student, RAD-DINO config at the pinned revision).

In [1]:
# ============================ CONFIG ============================
import os
os.environ.setdefault("CUBLAS_WORKSPACE_CONFIG", ":4096:8")      # determinism; set before CUDA starts
import json, glob, time, math, random, hashlib, shutil, warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.optimize import minimize
from scipy.special import expit
from scipy.stats import rankdata
warnings.filterwarnings("ignore")

# ---- what to run in THIS session ----
RUN_TEACHER  = True     # Session D1 (GPU, ~45 min): rebuild both teachers, QC, soft labels on train, go/no-go
RUN_STUDENT  = True     # Session D2 (GPU, ~3 h): train M10 on 0.5*y + 0.5*q - runs only if go/no-go says "go"
DEBUG        = False    # small subset + 1+1 student epochs; folders get a _debug tag and must never be reported

SEED = 42               # the only training seed of this study

PROTOCOL_VERSION = "2.2.0"
OUT_DIR   = os.environ.get("CXR_OUT_DIR", "/kaggle/working")
INPUT_DIR = os.environ.get("CXR_INPUT_DIR", "/kaggle/input")
DATA_ROOT = "/kaggle/input/datasets/nih-chest-xrays/data"      # NIH images + Data_Entry_2017.csv (sessions A/B)
NUM_WORKERS = 4

DATASET_ID, SPLIT_ID = "NIH_CXR14", "NIH_OFFICIAL_PV"
VAL_FRACTION, SPLIT_SEED = 0.15, 42                              # validation = 15 % of train_val patients
EXPECTED_COUNTS = {"train_val": 86524, "test": 25596, "total": 112120}
EXPECTED_LIST_SHA256 = {                                          # SHA-256 of sorted, LF-joined image ids
    "train_val_list.txt": "61fbe896321c1c1c8b75f3e4f3a08e4fef6486d95ef8a667c31d4d60dca6cb81",
    "test_list.txt":      "38ca5ef7f756092946f57c1a59faca882ed589a1ab1f72590b45dc06c6d5e1cc"}
EXPECTED_SPLIT_SHA256 = "b9f2fbc9aa061dafa76994494b92779651dc3ddd2ed3e7b82c70bbf5cf7071dd"   # split of the seed-42 runs

CLASSES = ["Atelectasis", "Cardiomegaly", "Effusion", "Infiltration", "Mass", "Nodule", "Pneumonia",
           "Pneumothorax", "Consolidation", "Edema", "Emphysema", "Fibrosis", "Pleural_Thickening", "Hernia"]
C = len(CLASSES)

ASL = dict(gamma_neg=4, gamma_pos=1, clip=0.05)
CNN_CFG = dict(
    kind="cnn", model_id="M08_CONVNEXT_TINY_ASL_384_CR", timm_name="convnext_tiny.fb_in22k_ft_in1k",
    img_size=384, mean=(0.485, 0.456, 0.406), std=(0.229, 0.224, 0.225),
    batch_size=32, warmup_epochs=1, finetune_epochs=8, patience=2,
    lr_head_warmup=1e-3, lr_head=1e-4, lr_backbone=1e-4,
    weight_decay=0.05, drop_path=0.1, use_ema=False, ema_decay=0.999, grad_clip_norm=1.0, asl=ASL, time_budget_h=8.0)
VIT_CFG = dict(
    kind="raddino", model_id="M09_RAD_DINO_518_CR", hf_name="microsoft/rad-dino", hf_revision="110cbc18d5133582e320b43d53bf5c44e410c936",
    img_size=518, mean=(0.5307,) * 3, std=(0.2583,) * 3,
    batch_size=12, warmup_epochs=2, finetune_epochs=4, patience=2, unfreeze_last=8, layer_decay=0.8,
    lr_head_warmup=1e-3, lr_head=2e-4, lr_backbone=2e-5,
    weight_decay=0.05, drop_path=0.1, use_ema=True, ema_decay=0.999, grad_clip_norm=1.0, asl=ASL, time_budget_h=9.0)

# ---- distillation (protocol section distillation, fixed before any distillation result) ----
STUDENT_CFG = dict(CNN_CFG, model_id="M10_CONVNEXT_TINY_KD_384")   # M08 recipe; only the training target changes
KD_LAMBDA = 0.5                  # target t = (1 - KD_LAMBDA) * y + KD_LAMBDA * q
GO_MAX_TRAIN_AUROC = 0.95        # go only if the teacher's train macro AUROC is below this
TEACHER_MEMBERS = {              # model_id -> (config, sha256 of checkpoint_best.pt)
    "M08_CONVNEXT_TINY_ASL_384_CR": (CNN_CFG, "eebd5f662d397d2fedeb5aa3fc70a2b59d70b05923e6c6712bf91049b9df70ad"),
    "M09_RAD_DINO_518_CR":          (VIT_CFG, "070f90c7cf44c54447a015791f5a18d0eb0d8b05bce070450b6c35e91008bd93")}
CAL_KEYS = {"M08_CONVNEXT_TINY_ASL_384_CR": "cnn", "M09_RAD_DINO_518_CR": "vit"}   # keys in v2.0.0 calibration.json

if DEBUG:                        # teachers keep their trained input size; only the student is shrunk
    STUDENT_CFG.update(img_size=224, warmup_epochs=1, finetune_epochs=1, batch_size=16)
    DATASET_ID += "_debug"
os.makedirs(OUT_DIR, exist_ok=True)

# ---------------- helpers ----------------
def sha256_bytes(b):
    return hashlib.sha256(b).hexdigest()

def sha256_file(path, chunk=1 << 20):
    h = hashlib.sha256()
    with open(path, "rb") as f:
        for blk in iter(lambda: f.read(chunk), b""):
            h.update(blk)
    return h.hexdigest()

def write_json(path, obj):
    with open(path, "w") as f:
        json.dump(obj, f, indent=2, default=str)

def save_table(df_, name):
    """CSV, UTF-8, full precision (rounding only when displayed)."""
    df_.to_csv(f"{OUT_DIR}/{name}.csv", index=False, float_format="%.17g", na_rep="")

def fmt(x, d=4):
    return "n/a" if x is None or not np.isfinite(x) else f"{np.round(x, d):.{d}f}"

def _walk(bases=(OUT_DIR, INPUT_DIR)):
    """os.walk over outputs and inputs, skipping the 112k-image folders."""
    for base in dict.fromkeys(bases):
        for root, dirs, files in os.walk(base):
            dirs[:] = [d for d in dirs if not d.startswith("images")]
            yield root, dirs, files

def find_file(fname):
    for root, _, files in _walk():
        if fname in files:
            return os.path.join(root, fname)
    return None

def run_folder_name(model_id):
    return f"{DATASET_ID}__{SPLIT_ID}__{model_id}__s{SEED}"

def find_run_dir(model_id):
    """Seed-42 run folder of a model that contains exported logits. Accepts the older names that carry a
    config hash (…__{model_id}__{hash}__s42__r0). More than one distinct folder is an error."""
    hits = {}
    for root, dirs, files in _walk():
        name = os.path.basename(root)
        if ("logits_test.npz" in files and name.startswith(f"{DATASET_ID}__{SPLIT_ID}__{model_id}__")
                and (name.endswith(f"__s{SEED}") or f"__s{SEED}__" in name) and "probe" not in name):
            hits.setdefault(name, root)                    # OUT_DIR is searched first
    assert len(hits) <= 1, f"several seed-{SEED} folders for {model_id}: {sorted(hits)} - attach only one"
    return next(iter(hits.values()), None)

## 1. Data and official split (identical code to the cascade notebook, so the split hash reproduces)

In [2]:
NEED_DATA = RUN_TEACHER or RUN_STUDENT
if NEED_DATA:
    from sklearn.model_selection import GroupShuffleSplit
    meta_csv = os.path.join(DATA_ROOT, "Data_Entry_2017.csv")
    if not os.path.exists(meta_csv):
        meta_csv = next((os.path.join(r, f) for r, _, fs in os.walk(INPUT_DIR) for f in fs if f == "Data_Entry_2017.csv"), meta_csv)
    assert os.path.exists(meta_csv), "Data_Entry_2017.csv not found - check DATA_ROOT"
    DATA_ROOT = os.path.dirname(meta_csv)
    png = glob.glob(os.path.join(DATA_ROOT, "images_*", "images", "*.png")) or \
          [os.path.join(r, f) for r, _, fs in os.walk(DATA_ROOT) for f in fs if f.endswith(".png")]
    df = pd.read_csv(meta_csv)
    df["path"] = df["Image Index"].map({os.path.basename(p): p for p in png})
    assert len(df) == EXPECTED_COUNTS["total"] and df.path.notna().all(), "images missing on disk"
    labels = df["Finding Labels"].str.split("|")
    for c in CLASSES:
        df[c] = labels.apply(lambda L, c=c: c in L).astype(np.float32)
    df = df.sort_values("Image Index").reset_index(drop=True)      # canonical order

    # ---- official list files (QC-01) ----
    def read_list(fname):
        p = os.path.join(DATA_ROOT, fname)
        p = p if os.path.exists(p) else find_file(fname)
        assert p, f"{fname} not found"
        return open(p).read().split()
    tv_ids, te_ids = read_list("train_val_list.txt"), read_list("test_list.txt")
    list_sha = {"train_val_list.txt": sha256_bytes("\n".join(sorted(tv_ids)).encode()),
                "test_list.txt": sha256_bytes("\n".join(sorted(te_ids)).encode())}
    assert (len(tv_ids), len(te_ids)) == (EXPECTED_COUNTS["train_val"], EXPECTED_COUNTS["test"]), "QC-01: list counts"
    assert list_sha == EXPECTED_LIST_SHA256, "QC-01: list files differ from the official lists"
    assert not set(tv_ids) & set(te_ids) and set(tv_ids) | set(te_ids) == set(df["Image Index"]), "QC-01: overlap/coverage"

    # ---- split (identical code to the one that produced the seed-42 runs) ----
    df["split"] = np.where(df["Image Index"].isin(set(te_ids)), "test", "trainval")
    tv_frame = df[df.split == "trainval"]
    _, va = next(GroupShuffleSplit(n_splits=1, test_size=VAL_FRACTION, random_state=SPLIT_SEED)
                 .split(tv_frame, groups=tv_frame["Patient ID"]))
    df.loc[tv_frame.index, "split"] = "train"
    df.loc[tv_frame.index[va], "split"] = "val"
    split_text = (df[["Image Index", "Patient ID", "split"]]
                  .rename(columns={"Image Index": "image_index", "Patient ID": "patient_id", "split": "partition"})
                  .sort_values("image_index").to_csv(index=False, lineterminator="\n"))
    SPLIT_SHA256 = sha256_bytes(split_text.encode("utf-8"))
    assert DEBUG or SPLIT_SHA256 == EXPECTED_SPLIT_SHA256, \
        f"split hash {SPLIT_SHA256[:12]} != {EXPECTED_SPLIT_SHA256[:12]}: library versions changed the validation carve-out"
    with open(f"{OUT_DIR}/split_{SPLIT_ID}.csv", "w", encoding="utf-8", newline="\n") as f:
        f.write(split_text)
    pats = {s: set(df.loc[df.split == s, "Patient ID"]) for s in ("train", "val", "test")}
    assert not (pats["train"] & pats["val"] or pats["train"] & pats["test"] or pats["val"] & pats["test"]), "QC-02: patient leakage"

    df_train, df_val, df_test = (df[df.split == s].reset_index(drop=True) for s in ("train", "val", "test"))
    if DEBUG:
        df_train, df_val, df_test = (d.sample(n, random_state=SPLIT_SEED).reset_index(drop=True)
                                     for d, n in ((df_train, 3000), (df_val, 1000), (df_test, 1000)))

    # ---- T1 dataset characteristics ----
    t1 = []
    for part, fr in (("train", df_train), ("val", df_val), ("test", df_test)):
        Y = fr[CLASSES].values
        base = dict(dataset_id=DATASET_ID, split_id=SPLIT_ID, partition=part, n_images=len(fr),
                    n_patients=fr["Patient ID"].nunique(), label_cardinality_mean=Y.sum(1).mean(),
                    n_all_negative_images=int((Y.sum(1) == 0).sum()),
                    images_per_patient_mean=len(fr) / fr["Patient ID"].nunique(),
                    view_PA_fraction=(fr["View Position"] == "PA").mean(), male_fraction=(fr["Patient Gender"] == "M").mean(),
                    age_median=fr["Patient Age"].median(), age_q25=fr["Patient Age"].quantile(.25),
                    age_q75=fr["Patient Age"].quantile(.75))
        t1.append(dict(base, label="ALL", n_positive=int(Y.sum()), prevalence=np.nan))
        t1 += [dict(base, label=c, n_positive=int(Y[:, i].sum()), prevalence=Y[:, i].mean()) for i, c in enumerate(CLASSES)]
    save_table(pd.DataFrame(t1), "T1_dataset_characteristics")
    print(f"split {SPLIT_ID} ({SPLIT_SHA256[:12]}) | train {len(df_train):,} | val {len(df_val):,} | test {len(df_test):,}")

split NIH_OFFICIAL_PV (b9f2fbc9aa06) | train 73,916 | val 12,608 | test 25,596


## 2. Model and training code (cascade notebook code; changes: pinned RAD-DINO revision, optional training targets)

In [3]:
import torch
import torch.nn as nn
import torchvision.transforms as T
from torch.utils.data import Dataset, DataLoader
from PIL import Image

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
USE_AMP = DEVICE.type == "cuda"

def set_determinism():
    torch.backends.cudnn.deterministic, torch.backends.cudnn.benchmark = True, False
    torch.use_deterministic_algorithms(True, warn_only=True)

def seed_everything(seed):
    random.seed(seed); np.random.seed(seed); torch.manual_seed(seed); torch.cuda.manual_seed_all(seed)

def worker_init_fn(worker_id):
    s = torch.initial_seed() % 2 ** 32
    np.random.seed(s); random.seed(s)

def autocast():
    return torch.autocast(device_type=DEVICE.type, dtype=torch.float16, enabled=USE_AMP)

class CXRDataset(Dataset):
    def __init__(self, paths, y, cfg, train):
        self.paths, self.y, s = paths, y.astype(np.float32), cfg["img_size"]
        self.tfm = (T.Compose([T.RandomResizedCrop(s, scale=(0.85, 1.0), ratio=(0.9, 1.1)), T.RandomHorizontalFlip(0.5),
                               T.RandomApply([T.RandomRotation(7)], p=0.5), T.ColorJitter(brightness=0.1, contrast=0.1),
                               T.ToTensor()]) if train else
                    T.Compose([T.Resize((s, s), interpolation=T.InterpolationMode.BILINEAR), T.ToTensor()]))
        self.norm = T.Normalize(cfg["mean"], cfg["std"])

    def __len__(self):
        return len(self.paths)

    def __getitem__(self, i):
        img = Image.open(self.paths[i]).convert("L")               # grayscale -> [0, 1] -> 3 channels
        return self.norm(self.tfm(img).repeat(3, 1, 1)), self.y[i]

def loader(frame, cfg, train, batch_size=None, y=None):
    """y: training targets in frame order (soft targets for the student); default = the hard labels."""
    g = torch.Generator().manual_seed(SEED)
    return DataLoader(CXRDataset(frame["path"].values, frame[CLASSES].values if y is None else y, cfg, train),
                      batch_size=batch_size or cfg["batch_size"], shuffle=train, num_workers=NUM_WORKERS,
                      pin_memory=USE_AMP, persistent_workers=NUM_WORKERS > 0, drop_last=False,
                      worker_init_fn=worker_init_fn, generator=g)

class AsymmetricLoss(nn.Module):
    """ASL (Ridnik et al., 2021)."""
    def __init__(self, gamma_neg=4, gamma_pos=1, clip=0.05, eps=1e-8):
        super().__init__()
        self.gn, self.gp, self.clip, self.eps = gamma_neg, gamma_pos, clip, eps

    def forward(self, logits, y):
        p = torch.sigmoid(logits.float())
        p_neg = (p - self.clip).clamp(min=0) if self.clip > 0 else p
        loss_pos = y * torch.log(p.clamp(min=self.eps)) * (1 - p).pow(self.gp)
        loss_neg = (1 - y) * torch.log((1 - p_neg).clamp(min=self.eps)) * p_neg.pow(self.gn)
        return -(loss_pos + loss_neg).sum(1).mean()

class CNNClassifier(nn.Module):
    def __init__(self, cfg, pretrained=True):
        super().__init__()
        import timm
        self.backbone = timm.create_model(cfg["timm_name"], pretrained=pretrained, num_classes=0, drop_path_rate=cfg["drop_path"])
        self.head = nn.Sequential(nn.Dropout(0.2), nn.Linear(self.backbone.num_features, C))

    def forward(self, x):
        return self.head(self.backbone(x))

    def backbone_param_groups(self, cfg):
        return [{"params": list(self.backbone.parameters()), "lr": cfg["lr_backbone"]}]

class RadDinoClassifier(nn.Module):
    """RAD-DINO ViT-B/14 + dual pooling (CLS ⊕ mean patch = 1536-d)."""
    def __init__(self, cfg, pretrained=True):
        super().__init__()
        from transformers import AutoModel, AutoConfig
        self.backbone = (AutoModel.from_pretrained(cfg["hf_name"], revision=cfg.get("hf_revision"), drop_path_rate=cfg["drop_path"]) if pretrained else
                         AutoModel.from_config(AutoConfig.from_pretrained(cfg["hf_name"], revision=cfg.get("hf_revision"), drop_path_rate=cfg["drop_path"])))
        self.n_reg = getattr(self.backbone.config, "num_register_tokens", 0)
        self.head = nn.Sequential(nn.Dropout(0.2), nn.Linear(2 * self.backbone.config.hidden_size, C))

    def forward(self, x):
        h = self.backbone(pixel_values=x).last_hidden_state
        return self.head(torch.cat([h[:, 0], h[:, 1 + self.n_reg:].mean(1)], dim=1))

    def backbone_param_groups(self, cfg):
        layers, k = self.backbone.encoder.layer, cfg["unfreeze_last"]
        n = len(layers)
        groups = [{"params": list(self.backbone.layernorm.parameters()), "lr": cfg["lr_backbone"]}]
        groups += [{"params": list(layers[i].parameters()), "lr": cfg["lr_backbone"] * cfg["layer_decay"] ** (n - 1 - i)}
                   for i in range(n - k, n)]
        return groups

def build_model(cfg, pretrained=True):
    seed_everything(SEED)                                          # seeded head initialisation
    return (CNNClassifier(cfg, pretrained) if cfg["kind"] == "cnn" else RadDinoClassifier(cfg, pretrained)).to(DEVICE)

@torch.no_grad()
def predict_logits(model, dl):
    model.eval()
    out = []
    for x, _ in dl:
        with autocast():
            out.append(model(x.to(DEVICE, non_blocking=True)).float().cpu())
    return torch.cat(out).numpy().astype(np.float32)

def configure_phase(model, cfg, phase):
    """warmup = head only; finetune = head + (part of) the backbone."""
    for p in model.parameters():
        p.requires_grad = False
    groups = [{"params": list(model.head.parameters()), "lr": cfg["lr_head_warmup"] if phase == "warmup" else cfg["lr_head"]}]
    if phase == "finetune":
        groups += model.backbone_param_groups(cfg)
    for g in groups:
        for p in g["params"]:
            p.requires_grad = True
    return torch.optim.AdamW(groups, weight_decay=cfg["weight_decay"])

def cosine_schedule(opt, total_steps, warmup_frac=0.03):
    w = max(1, int(total_steps * warmup_frac))
    f = lambda s: (s + 1) / w if s < w else 0.5 * (1 + math.cos(math.pi * (s - w) / max(1, total_steps - w)))
    return torch.optim.lr_scheduler.LambdaLR(opt, f)

def val_macro_auroc(y, s):
    from sklearn.metrics import roc_auc_score
    return float(np.mean([roc_auc_score(y[:, c], s[:, c]) for c in range(C) if 0 < y[:, c].sum() < len(y)]))

def train_model(cfg, exp_dir, model, y_train=None):
    loss_fn = AsymmetricLoss(**cfg["asl"])
    train_dl, val_dl = loader(df_train, cfg, True, y=y_train), loader(df_val, cfg, False, cfg["batch_size"] * 2)
    yv = df_val[CLASSES].values
    scaler = torch.amp.GradScaler("cuda", enabled=USE_AMP)
    plan = [("warmup", i) for i in range(cfg["warmup_epochs"])] + [("finetune", i) for i in range(cfg["finetune_epochs"])]
    best, best_epoch, bad, ema, phase_now, t0, ep_times, truncated = -1.0, 0, 0, None, None, time.time(), [], False
    for ep, (phase, _) in enumerate(plan, 1):
        if ep_times and (time.time() - t0 + max(ep_times)) / 3600 > cfg["time_budget_h"]:
            print("Stopping: next epoch would exceed the time budget -> truncated=true"); truncated = True
            break
        if phase != phase_now:
            phase_now, bad = phase, 0
            opt = configure_phase(model, cfg, phase)
            n_ep = cfg["warmup_epochs"] if phase == "warmup" else cfg["finetune_epochs"]
            sched = cosine_schedule(opt, n_ep * len(train_dl))
            if phase == "finetune" and cfg["use_ema"]:
                from timm.utils import ModelEmaV2
                ema = ModelEmaV2(model, decay=cfg["ema_decay"])
        if USE_AMP:
            torch.cuda.reset_peak_memory_stats()
        model.train()
        te, run_loss = time.time(), 0.0
        for i, (x, y) in enumerate(train_dl):
            x, y = x.to(DEVICE, non_blocking=True), y.to(DEVICE, non_blocking=True)
            with autocast():
                logits = model(x)
            loss = loss_fn(logits, y)
            opt.zero_grad(set_to_none=True)
            scaler.scale(loss).backward()
            scaler.unscale_(opt)
            torch.nn.utils.clip_grad_norm_([p for p in model.parameters() if p.requires_grad], cfg["grad_clip_norm"])
            scaler.step(opt); scaler.update(); sched.step()
            if ema is not None:
                ema.update(model)
            run_loss += loss.item()
            if i % 500 == 0:
                print(f"  epoch {ep} step {i}/{len(train_dl)} loss {loss.item():.4f}", flush=True)
        eval_model = ema.module if ema is not None else model                 # EMA weights are evaluated (M09)
        zv = predict_logits(eval_model, val_dl)
        auc = val_macro_auroc(yv, zv)
        ep_times.append(time.time() - te)
        row = dict(epoch=ep, phase=phase, train_loss=run_loss / len(train_dl),
                   val_loss=float(loss_fn(torch.as_tensor(zv), torch.as_tensor(yv))), val_macro_auroc=auc,
                   epoch_seconds=ep_times[-1], peak_gpu_mem_mb=torch.cuda.max_memory_allocated() / 2 ** 20 if USE_AMP else np.nan)
        log = f"{exp_dir}/train_log.csv"
        pd.DataFrame([row]).to_csv(log, mode="a", header=not os.path.exists(log), index=False)
        print(f"Epoch {ep} [{phase}] loss {row['train_loss']:.4f} | val macro AUROC {auc:.4f} | {ep_times[-1] / 60:.1f} min", flush=True)
        if auc > best:                                                        # strict '>' keeps the earlier epoch on ties
            best, best_epoch, bad = auc, ep, 0
            torch.save(eval_model.state_dict(), f"{exp_dir}/checkpoint_best.pt")
        elif phase == "finetune":
            bad += 1
            if bad >= cfg["patience"]:
                print("Early stopping."); break
    model.load_state_dict(torch.load(f"{exp_dir}/checkpoint_best.pt", map_location=DEVICE))
    return model, dict(best_val_macro_auroc=best, best_epoch=best_epoch, epochs_trained=len(ep_times),
                       truncated=truncated, train_wall_clock_h=(time.time() - t0) / 3600)

def run_stage(cfg, y_train=None, extra_info=None):
    """Train one model with SEED and export val/test logits. Skips if the seed-42 logits already exist.
    y_train: training targets in df_train order (default: hard labels); extra_info: stored in config.json."""
    existing = find_run_dir(cfg["model_id"])
    if existing:
        print(f"{cfg['model_id']}: logits already at {existing} - skipping."); return
    exp_dir = f"{OUT_DIR}/{run_folder_name(cfg['model_id'])}"
    os.makedirs(exp_dir, exist_ok=True)
    set_determinism()
    model = build_model(cfg)
    model, info = train_model(cfg, exp_dir, model, y_train)
    for part, frame in (("val", df_val), ("test", df_test)):
        z = predict_logits(model, loader(frame, cfg, False, cfg["batch_size"] * 2))
        np.savez(f"{exp_dir}/logits_{part}.npz", image_index=frame["Image Index"].to_numpy(dtype=str),
                 patient_id=frame["Patient ID"].to_numpy(dtype=np.int64), logits=z,
                 labels=frame[CLASSES].to_numpy(dtype=np.uint8), label_order=json.dumps(CLASSES),
                 split_id=SPLIT_ID, split_sha256=SPLIT_SHA256, model_id=cfg["model_id"], seed=SEED, eval_set=part)
    import timm, transformers
    write_json(f"{exp_dir}/config.json", dict(
        model_id=cfg["model_id"], seed=SEED, split_id=SPLIT_ID, split_sha256=SPLIT_SHA256, protocol_version=PROTOCOL_VERSION,
        model_cfg=cfg, train_info=info, checkpoint_sha256=sha256_file(f"{exp_dir}/checkpoint_best.pt"),
        software=dict(torch=torch.__version__, timm=timm.__version__, transformers=transformers.__version__),
        gpu=torch.cuda.get_device_name(0) if USE_AMP else "cpu", **(extra_info or {})))
    print(f"{cfg['model_id']}: done ({info['train_wall_clock_h']:.2f} h, best epoch {info['best_epoch']})")


## 3. Session D1: rebuild the teachers, check them, write soft labels, apply the go/no-go rule

QC-D1: checkpoint hashes. QC-D2: each rebuilt teacher reproduces its saved validation logits. QC-D3: soft labels only for train images. Go if the teacher's train macro AUROC is < 0.95.

In [4]:
# ===================== Session D1: teacher soft labels + go/no-go (GPU) =====================
TEACHER_DIR = f"{OUT_DIR}/teacher"
QC_D = {}

def find_checkpoint(model_id, sha):
    """checkpoint_best.pt in a seed-42 run folder of model_id whose sha256 equals the recorded hash (QC-D1)."""
    seen = []
    for root, _, files in _walk():
        if "checkpoint_best.pt" in files and os.path.basename(root).startswith(f"{DATASET_ID.replace('_debug', '')}__{SPLIT_ID}__{model_id}__"):
            p = os.path.join(root, "checkpoint_best.pt")
            h = sha256_file(p)
            if h == sha:
                return p
            seen.append((p, h[:12]))
    raise FileNotFoundError(f"QC-D1: no checkpoint_best.pt for {model_id} with sha256 {sha[:12]}... - found {seen}")

def platt(z, model_id):
    c = CAL[CAL_KEYS[model_id]]
    return expit(np.asarray(c["a"]) * z + np.asarray(c["b"]))

def ids(frame):
    return frame["Image Index"].to_numpy(dtype=str)

if RUN_TEACHER:
    assert USE_AMP, "session D1 needs a GPU"
    cal_path = find_file("calibration.json")
    assert cal_path, "calibration.json (v2.0.0, results_s42) not found - attach it as input"
    CAL = json.load(open(cal_path))
    os.makedirs(TEACHER_DIR, exist_ok=True)
    p_train, p_val = {}, {}
    for model_id, (tcfg, sha) in TEACHER_MEMBERS.items():
        t0 = time.time()
        model = build_model(tcfg, pretrained=False)
        model.load_state_dict(torch.load(find_checkpoint(model_id, sha), map_location=DEVICE))
        model.eval()
        QC_D[f"QC-D1 {model_id}"] = f"pass ({sha[:12]})"
        bs = tcfg["batch_size"] * 2

        # QC-D2: the rebuilt teacher must reproduce its saved v2.0.0 validation logits
        zv = predict_logits(model, loader(df_val, tcfg, False, bs))
        np.savez(f"{TEACHER_DIR}/{model_id}_logits_val_rebuilt.npz", image_index=ids(df_val), logits=zv)
        if DEBUG:
            QC_D[f"QC-D2 {model_id}"] = "skipped (DEBUG subset)"
        else:
            saved = np.load(os.path.join(find_run_dir(model_id), "logits_val.npz"))
            assert (saved["image_index"] == ids(df_val)).all(), f"QC-D2 {model_id}: validation image order differs"
            yv = df_val[CLASSES].values
            corr = min(np.corrcoef(zv[:, c], saved["logits"][:, c])[0, 1] for c in range(C))
            d_auc = abs(val_macro_auroc(yv, zv) - val_macro_auroc(yv, saved["logits"]))
            ok = corr >= 0.999 and d_auc <= 0.001
            QC_D[f"QC-D2 {model_id}"] = f"{'pass' if ok else 'FAIL'} (min per-class corr {corr:.5f}, |d macro AUROC| {d_auc:.5f})"
            assert ok, QC_D[f"QC-D2 {model_id}"]

        zt = predict_logits(model, loader(df_train, tcfg, False, bs))
        np.savez(f"{TEACHER_DIR}/{model_id}_logits_train.npz", image_index=ids(df_train),
                 patient_id=df_train["Patient ID"].to_numpy(dtype=np.int64), logits=zt,
                 labels=df_train[CLASSES].to_numpy(dtype=np.uint8), label_order=json.dumps(CLASSES),
                 checkpoint_sha256=sha, split_id=SPLIT_ID, split_sha256=SPLIT_SHA256)
        p_train[model_id], p_val[model_id] = platt(zt, model_id), platt(zv, model_id)
        print(f"{model_id}: {QC_D[f'QC-D2 {model_id}']} | train + val pass {(time.time() - t0) / 60:.1f} min", flush=True)
        del model
        torch.cuda.empty_cache()

    # soft labels: train images only, df_train order (QC-D3)
    q = np.mean([p_train[m] for m in TEACHER_MEMBERS], axis=0).astype(np.float32)
    leak = set(ids(df_train)) & (set(ids(df_val)) | set(ids(df_test)))
    QC_D["QC-D3"] = "pass" if len(q) == len(df_train) and not leak else f"FAIL ({len(leak)} val/test images)"
    assert QC_D["QC-D3"] == "pass", QC_D["QC-D3"]
    np.savez(f"{OUT_DIR}/soft_labels_train.npz", image_index=ids(df_train), q=q,
             labels=df_train[CLASSES].to_numpy(dtype=np.uint8), label_order=json.dumps(CLASSES),
             kd_lambda=KD_LAMBDA, split_sha256=SPLIT_SHA256)

    # go / no-go (protocol distillation.go_no_go)
    yt, yv = df_train[CLASSES].values, df_val[CLASSES].values
    auc = {f"{m} train": val_macro_auroc(yt, p_train[m]) for m in TEACHER_MEMBERS}
    auc |= {f"{m} val": val_macro_auroc(yv, p_val[m]) for m in TEACHER_MEMBERS}
    auc["teacher train"] = val_macro_auroc(yt, q)
    auc["teacher val"] = val_macro_auroc(yv, np.mean([p_val[m] for m in TEACHER_MEMBERS], axis=0))
    decision = "go" if auc["teacher train"] < GO_MAX_TRAIN_AUROC else "stop"
    write_json(f"{OUT_DIR}/distill_go_no_go.json", dict(
        protocol_version=PROTOCOL_VERSION, rule=f"go if teacher train macro AUROC < {GO_MAX_TRAIN_AUROC}",
        decision=decision, macro_auroc=auc, split_sha256=SPLIT_SHA256, debug=DEBUG,
        mean_soft_label_on_positives=float(q[yt == 1].mean()), mean_soft_label_on_negatives=float(q[yt == 0].mean())))
    write_json(f"{OUT_DIR}/qc_session_D.json", QC_D)
    print(pd.Series(auc).round(4).to_string())
    print(f"\nGo/no-go: teacher train macro AUROC {auc['teacher train']:.4f} "
          f"(val {auc['teacher val']:.4f}) -> {decision.upper()}")


M08_CONVNEXT_TINY_ASL_384_CR: pass (min per-class corr 1.00000, |d macro AUROC| 0.00000) | train + val pass 14.7 min


config.json:   0%|          | 0.00/879 [00:00<?, ?B/s]

M09_RAD_DINO_518_CR: pass (min per-class corr 1.00000, |d macro AUROC| 0.00000) | train + val pass 35.3 min
M08_CONVNEXT_TINY_ASL_384_CR train    0.9018
M09_RAD_DINO_518_CR train             0.9124
M08_CONVNEXT_TINY_ASL_384_CR val      0.8598
M09_RAD_DINO_518_CR val               0.8651
teacher train                         0.9165
teacher val                           0.8735

Go/no-go: teacher train macro AUROC 0.9165 (val 0.8735) -> GO


## 4. Session D2: train the student M10

Same recipe as M08; target = 0.5 x hard label + 0.5 x teacher soft label; model selection on validation macro AUROC against hard labels. QC-D4: the student's val/test exports match M08's.

In [5]:
# ===================== Session D2: train the student M10 (GPU) =====================
def input_or_output(fname):
    p = f"{OUT_DIR}/{fname}"
    return p if os.path.exists(p) else find_file(fname)

if RUN_STUDENT:
    gng = json.load(open(input_or_output("distill_go_no_go.json")))
    assert gng["debug"] == DEBUG, "go/no-go file comes from a different DEBUG setting"
    if gng["decision"] != "go":
        print(f"Go/no-go = {gng['decision'].upper()} (teacher train macro AUROC {gng['macro_auroc']['teacher train']:.4f}) "
              "- the student is not trained (protocol distillation.go_no_go).")
    else:
        soft = np.load(input_or_output("soft_labels_train.npz"))
        assert (soft["image_index"] == df_train["Image Index"].to_numpy(dtype=str)).all(), "QC-D3: soft labels not in df_train order"
        assert str(soft["split_sha256"]) == SPLIT_SHA256, "QC-D3: soft labels come from another split"
        y_mix = ((1 - KD_LAMBDA) * df_train[CLASSES].to_numpy(dtype=np.float32) + KD_LAMBDA * soft["q"]).astype(np.float32)
        run_stage(STUDENT_CFG, y_train=y_mix, extra_info=dict(distillation=dict(
            teacher="(p_M08 + p_M09) / 2, v2.0.0 Platt calibration", target="(1 - kd_lambda) * y + kd_lambda * q",
            kd_lambda=KD_LAMBDA, teacher_checkpoint_sha256={m: s for m, (_, s) in TEACHER_MEMBERS.items()},
            go_no_go=gng)))

        # QC-D4: student's val/test exports must match M08's images, order, labels and split
        qc_path = input_or_output("qc_session_D.json")
        QC_D = json.load(open(qc_path)) if qc_path else {}
        if DEBUG:
            QC_D["QC-D4"] = "skipped (DEBUG subset)"
        else:
            m08, m10 = find_run_dir(CNN_CFG["model_id"]), find_run_dir(STUDENT_CFG["model_id"])
            bad = []
            for part in ("val", "test"):
                a, b = np.load(f"{m08}/logits_{part}.npz"), np.load(f"{m10}/logits_{part}.npz")
                if not ((a["image_index"] == b["image_index"]).all() and (a["labels"] == b["labels"]).all()
                        and str(a["split_sha256"]) == str(b["split_sha256"])):
                    bad.append(part)
            QC_D["QC-D4"] = "pass" if not bad else f"FAIL ({bad})"
        write_json(f"{OUT_DIR}/qc_session_D.json", QC_D)
        print(QC_D)


model.safetensors:   0%|          | 0.00/114M [00:00<?, ?B/s]

  epoch 1 step 0/2310 loss 0.9380
  epoch 1 step 500/2310 loss 0.4249
  epoch 1 step 1000/2310 loss 0.4103
  epoch 1 step 1500/2310 loss 0.3462
  epoch 1 step 2000/2310 loss 0.3382
Epoch 1 [warmup] loss 0.3927 | val macro AUROC 0.7511 | 14.5 min
  epoch 2 step 0/2310 loss 0.3965
  epoch 2 step 500/2310 loss 0.2710
  epoch 2 step 1000/2310 loss 0.3682
  epoch 2 step 1500/2310 loss 0.2361
  epoch 2 step 2000/2310 loss 0.2857
Epoch 2 [finetune] loss 0.3445 | val macro AUROC 0.8426 | 19.9 min
  epoch 3 step 0/2310 loss 0.4080
  epoch 3 step 500/2310 loss 0.2741
  epoch 3 step 1000/2310 loss 0.3241
  epoch 3 step 1500/2310 loss 0.3375
  epoch 3 step 2000/2310 loss 0.2770
Epoch 3 [finetune] loss 0.3195 | val macro AUROC 0.8570 | 19.7 min
  epoch 4 step 0/2310 loss 0.4062
  epoch 4 step 500/2310 loss 0.2366
  epoch 4 step 1000/2310 loss 0.3244
  epoch 4 step 1500/2310 loss 0.3333
  epoch 4 step 2000/2310 loss 0.3116
Epoch 4 [finetune] loss 0.3122 | val macro AUROC 0.8622 | 19.8 min
  epoch 5 

## 5. Session D3: evaluation

To be added after D2: calibration and thresholds on validation, test metrics with patient-level bootstrap, comparisons Distilled vs CNN / RAD-DINO / Average (Holm over 3), tables T10-T11.